# NVIDIA Synthetic Video Detector - Python Notebook

This notebook demonstrates how to use the NVIDIA Synthetic Video Detector service with Python. The service analyzes videos to detect whether they are synthetic/fake or real using a DINO-based deep learning model.

## Overview

The service processes video files and provides detection results with probability scores:

- **Service Integration**: Connect to NVIDIA Synthetic Video Detector services
- **Video Analysis**: Detect synthetic/fake content in videos
- **Two Input Modes**: Stream a local file, or pass a pre-signed HTTPS URL (S3 ingest) and let the NIM fetch the object itself
- **Input Preview**: Render the input video inline before inference, for both local files and pre-signed URLs
- **Per-frame Results**: Get detailed detection scores for each video frame
- **Visualization**: Plot probability scores over time with threshold lines
- **CSV Export**: Save detailed results for further analysis

## Requirements

- **Input Video**: MP4/QuickTime (`.mp4`, `.mov`, `.m4v`), Matroska (`.mkv`), and WebM (`.webm`) containers with H.264/AVC, H.265/HEVC, AV1, VP8, or VP9 video (audio optional). Variable Frame Rate (VFR) input is supported. Decoding runs on the GPU via NVDEC, so 4:2:2 chroma and 10-bit H.264 are not decodable; AV1 requires an Ampere or newer GPU. See the client README for the full matrix.
- **Output**: Detection verdict (REAL vs SYNTHETIC/FAKE) with confidence scores
- **Service**: Access to a running NVIDIA Synthetic Video Detector service instance


## Installation

### Requirements:
- Python 3.10+ or later
- pip package manager
- gRPC and matplotlib dependencies from the requirements.txt file

Run the cell below to install all required packages:


In [ ]:
%pip install -r ../../requirements.txt

## Service Configuration

Configure the connection to your NVIDIA Synthetic Video Detector service. The service can be running on your machine or on a remote server accessible from your environment.


In [ ]:
import os
import sys
import pathlib

# Setup paths for NVIDIA Synthetic Video Detector modules
SCRIPT_PATH = str(pathlib.Path().resolve())
print(f"Notebook path: {SCRIPT_PATH}")

sys.path.append(os.path.join(SCRIPT_PATH, "../../scripts/v2"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../interfaces"))

# Service connection configuration
SERVICE_HOST = "localhost"  # Update to your service host
SERVICE_PORT = 8001         # Update to your service port
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print(f"Python paths configured for NVIDIA Synthetic Video Detector modules")


## Import Libraries

Import the required libraries for gRPC communication, visualization, and the NVIDIA Synthetic Video Detector service modules. This cell also defines a sigmoid utility function used to convert model logits into probability scores.

In [ ]:
import grpc
import time
import math
from typing import Iterator, List, Optional, Tuple
from urllib.parse import urlsplit
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64
from io import BytesIO

# Import NVIDIA Synthetic Video Detector modules
from constants import DATA_CHUNK_SIZE
from nvidia.ai4m.ingest.v1 import presigned_url_pb2 as ingest_pb2
from nvidia.ai4m.syntheticvideodetector.v1 import (
    syntheticvideodetector_pb2,
    syntheticvideodetector_pb2_grpc,
)

# Maps between the friendly strings used below and the generated protobuf enum
# values. Mirrors scripts/v2/synthetic-video-detector.py so the notebook and
# the CLI build identical pre-signed URL requests.
_CHECKSUM_TYPE = {
    "md5": ingest_pb2.CHECKSUM_TYPE_MD5,
    "sha256": ingest_pb2.CHECKSUM_TYPE_SHA256,
    "sha512": ingest_pb2.CHECKSUM_TYPE_SHA512,
}
_CHECKSUM_HEX_FIELD = {"md5": "md5_hex", "sha256": "sha256_hex", "sha512": "sha512_hex"}
_PROVIDER = {
    "unspecified": ingest_pb2.URL_PROVIDER_UNSPECIFIED,
    "s3": ingest_pb2.URL_PROVIDER_S3,
}


def sigmoid(x: float) -> float:
    """Convert logit to probability using sigmoid function.
    
    Args:
        x: Logit value
        
    Returns:
        Probability value between 0 and 1
    """
    try:
        return 1.0 / (1.0 + math.exp(-x))
    except OverflowError:
        return 0.0 if x < 0 else 1.0


print("All libraries imported successfully!")

## Helper Functions

Define functions for processing video data and communicating with the NVIDIA Synthetic Video Detector service. These functions handle the bi-directional gRPC streaming protocol used by the service.

### Function Overview

The NVIDIA Synthetic Video Detector service uses bi-directional gRPC streaming:

1. **Request Generator**: Streams video data in chunks to the service
2. **Response Processor**: Processes incoming detection results (per-frame results, final results, keepalive messages)
3. **Visualization**: Plots probability scores over time with threshold lines


In [ ]:
def fmt_elapsed(sec: float) -> str:
    """Format elapsed time in MM:SS format.
    
    Args:
        sec: Elapsed time in seconds
        
    Returns:
        Formatted string in MM:SS format
    """
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"


def redact_url(url: str) -> str:
    """Return a log-safe URL: scheme://host/path with the query dropped.

    Pre-signed URLs carry their credentials in the query string, so only the
    redacted form is printed.

    Args:
        url: The (possibly credential-bearing) URL to redact

    Returns:
        The URL without its query string
    """
    parts = urlsplit(url)
    return f"{parts.scheme}://{parts.hostname or ''}{parts.path}"


def build_presigned_url(
    url: str,
    url_provider: str = "unspecified",
    checksum_type: str = "sha256",
    checksum: Optional[str] = None,
    no_verify_checksum: bool = False,
    expected_size: Optional[int] = None,
    content_type: Optional[str] = None,
    expires_at_unix_ms: Optional[int] = None,
) -> ingest_pb2.PresignedUrl:
    """Build a PresignedUrl message for the S3 ingest path.

    This is a thin pass-through of the supplied fields. All validation (HTTPS,
    host allowlist, expiry, SSRF, checksum, size, content-type) is performed by
    the NIM, so the client stays lean.

    Args:
        url: Absolute HTTPS pre-signed GET URL
        url_provider: 'unspecified' or 's3' (enables SigV4 URL-shape checks)
        checksum_type: 'md5', 'sha256', or 'sha512'
        checksum: Expected lowercase hex digest of the object
        no_verify_checksum: Ask the server to skip checksum verification
        expected_size: Expected Content-Length in bytes
        content_type: Expected Content-Type, e.g. 'video/mp4'
        expires_at_unix_ms: Client assertion of URL expiry (epoch ms)

    Returns:
        A populated PresignedUrl protobuf message
    """
    obj = ingest_pb2.PresignedUrl(url=url)
    obj.url_provider = _PROVIDER[url_provider]

    if no_verify_checksum:
        obj.verify_checksum = False
    elif checksum:
        obj.verify_checksum = True
        digest = ingest_pb2.FileChecksum(type=_CHECKSUM_TYPE[checksum_type])
        setattr(digest, _CHECKSUM_HEX_FIELD[checksum_type], checksum.lower())
        obj.checksum.CopyFrom(digest)

    if expected_size is not None:
        obj.expected_content_length_bytes = expected_size
    if content_type:
        obj.content_type = content_type
    if expires_at_unix_ms is not None:
        obj.expires_at_unix_ms = expires_at_unix_ms

    return obj


def preview_input_video(
    video_filepath: Optional[str] = None,
    presigned_url: Optional[str] = None,
    width: int = 640,
) -> None:
    """Render an inline preview of the input video.

    For a pre-signed URL the browser streams the object directly, so no
    download happens in Python. Note that the full signed URL is embedded in
    the rendered HTML, so clear the cell output before sharing the notebook.

    Args:
        video_filepath: Local video path (byte-streaming path)
        presigned_url: Pre-signed HTTPS URL (S3 ingest path)
        width: Player width in pixels
    """
    if presigned_url:
        print(f"Input preview (URL): {redact_url(presigned_url)}")
        display(HTML(
            f'<video src="{presigned_url}" controls width="{width}" '
            'style="max-width:100%;border-radius:6px"></video>'
        ))
    elif video_filepath and os.path.exists(video_filepath):
        print(f"Input preview (file): {video_filepath}")
        display(Video(video_filepath, embed=True, width=width))
    else:
        print("No preview available for the given input.")


def generate_request_for_inference(
    video_filepath: Optional[str] = None,
    chunk_size: int = DATA_CHUNK_SIZE,
    presigned_url: Optional[str] = None,
    url_provider: str = "unspecified",
    checksum_type: str = "sha256",
    checksum: Optional[str] = None,
    no_verify_checksum: bool = False,
    expected_size: Optional[int] = None,
    content_type: Optional[str] = None,
    expires_at_unix_ms: Optional[int] = None,
) -> Iterator[syntheticvideodetector_pb2.DetectSyntheticVideoRequest]:
    """Generate requests for the NVIDIA Synthetic Video Detector service.
    
    Handles both input modes behind one generator (mirroring the CLI):
    a pre-signed URL is sent as a single request and the NIM fetches the
    object itself; otherwise the local file is streamed as byte chunks.
    
    Args:
        video_filepath: Path to the input video file (byte-streaming path)
        chunk_size: Size of data chunks to send (default: 1MB)
        presigned_url: Pre-signed HTTPS URL; takes precedence when set
        url_provider: 'unspecified' or 's3'
        checksum_type: 'md5', 'sha256', or 'sha512'
        checksum: Expected lowercase hex digest of the object
        no_verify_checksum: Ask the server to skip checksum verification
        expected_size: Expected Content-Length in bytes
        content_type: Expected Content-Type, e.g. 'video/mp4'
        expires_at_unix_ms: Client assertion of URL expiry (epoch ms)
    
    Yields:
        DetectSyntheticVideoRequest messages for the selected input mode
        
    Raises:
        ValueError: If neither or both input modes are supplied
        FileNotFoundError: If input file doesn't exist
        IOError: If input file cannot be read
    """
    
    if presigned_url and video_filepath:
        raise ValueError("Provide either video_filepath or presigned_url, not both.")
    if not presigned_url and not video_filepath:
        raise ValueError("One of video_filepath or presigned_url is required.")
    
    # Pre-signed URL (S3 ingest): one request, the NIM downloads the object
    if presigned_url:
        print(f"Sending pre-signed URL request: {redact_url(presigned_url)}")
        yield syntheticvideodetector_pb2.DetectSyntheticVideoRequest(
            presigned_url=build_presigned_url(
                url=presigned_url,
                url_provider=url_provider,
                checksum_type=checksum_type,
                checksum=checksum,
                no_verify_checksum=no_verify_checksum,
                expected_size=expected_size,
                content_type=content_type,
                expires_at_unix_ms=expires_at_unix_ms,
            )
        )
        return
    
    # Byte streaming: validate input file exists
    if not os.path.exists(video_filepath):
        raise FileNotFoundError(f"Video file not found: {video_filepath}")
    
    file_size = os.path.getsize(video_filepath)
    print(f"Video file size: {file_size / (1024*1024):.2f} MB")
    
    # Send video data in chunks with progress tracking
    try:
        with open(video_filepath, "rb") as video_file:
            while True:
                chunk = video_file.read(chunk_size)
                if not chunk:
                    break
                yield syntheticvideodetector_pb2.DetectSyntheticVideoRequest(
                    video_file_data=chunk
                )
        print("Upload complete!")
    except IOError as e:
        raise IOError(f"Failed to read video file: {e}") from e


def process_detection_response(
    response_iter: Iterator[syntheticvideodetector_pb2.DetectSyntheticVideoResponse],
    show_progress: bool = True
) -> Tuple[List[dict], dict]:
    """Process detection responses from the service.
    
    Args:
        response_iter: Iterator of response messages from the service
        show_progress: Whether to show progress updates
    
    Returns:
        Tuple of (frame_results, final_result) where:
        - frame_results: List of dicts with 'index', 'logit', and 'probability' for each frame
        - final_result: Dict with final aggregated results
    """
    
    frame_results = []
    final_result = None
    start_time = time.time()
    
    if show_progress:
        print("Processing detection responses...")
    
    try:
        for response in response_iter:
            if response.HasField("service_info"):
                # One-shot provenance banner emitted first in the stream
                if show_progress:
                    si = response.service_info
                    print(
                        f"Server: {si.nim_version} | "
                        f"Model: {si.model_version} | ReqID: {si.request_id}"
                    )

            elif response.HasField("frame_results"):
                # One FrameResultBatch per inference batch
                for frame in response.frame_results.results:
                    probability = sigmoid(frame.logit)  # Convert logit to probability
                    frame_results.append({
                        'index': frame.frame_id,
                        'logit': frame.logit,
                        'probability': probability
                    })

            elif response.HasField("final_result"):
                final_result = {
                    'logit': response.final_result.mean_logit,
                    'probability': response.final_result.synthetic_probability,
                    'total_frames': response.final_result.total_frames,
                }

            elif response.HasField("keepalive"):
                # Keepalive message - continue processing
                pass
        if show_progress and frame_results:
            elapsed = fmt_elapsed(time.time() - start_time)
            print(f"Processed {len(frame_results)} frames in {elapsed}")
    except Exception as e:
        print(f"\nError processing responses: {e}")
        raise
    
    return frame_results, final_result


print("Helper functions defined successfully!")

## Visualization Function

Create a function to plot probability scores over time with a threshold line to visualize the detection results.


In [ ]:
def plot_detection_results(
    clip_results: List[dict],
    final_result: dict,
    video_filepath: str = None,
    threshold: float = 0.44,
    figsize: tuple = (14, 6),
    video_name: str = "Video"
):
    """Plot per-frame probability scores over time and display the video being tested.
    
    Args:
        clip_results: List of dicts with 'index', 'logit', and 'probability' per frame
        final_result: Dict with final aggregated results
        video_filepath: Path to the video file to display
        threshold: Threshold value for synthetic/real classification (default: 0.44)
        figsize: Figure size as (width, height) tuple
        video_name: Name of the video for the plot title
    """
    
    if not clip_results:
        print("No frame results to plot")
        return
    
    # Extract data for plotting
    indices = [clip['index'] for clip in clip_results]
    probabilities = [clip['probability'] for clip in clip_results]
    
    # Use frame indices for x-axis
    x_data = indices
    x_label = 'Frame Index (Frame Number)'
    # Create figure with one subplot
    fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))
    
    # Plot: Probability over time
    ax1.plot(x_data, probabilities, 'b-', linewidth=2, label='Probability', alpha=0.7)
    ax1.axhline(y=threshold, color='r', linestyle='--', linewidth=2, 
                label=f'Threshold ({threshold})')
    
    # Add final probability as horizontal line
    if final_result:
        ax1.axhline(y=final_result['probability'], color='g', linestyle='-', 
                   linewidth=2, alpha=0.5, label=f"Final Prob ({final_result['probability']:.3f})")
    
    # Fill areas above and below threshold
    ax1.fill_between(x_data, probabilities, threshold, 
                     where=[p >= threshold for p in probabilities],
                     alpha=0.3, color='red', label='Synthetic region')
    ax1.fill_between(x_data, probabilities, threshold,
                     where=[p < threshold for p in probabilities],
                     alpha=0.3, color='green', label='Real region')
    
    ax1.set_xlabel(x_label, fontsize=12)
    ax1.set_ylabel('Probability (Synthetic)', fontsize=12)
    ax1.set_title(f'Detection Probability Over Time - {video_name}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best')
    ax1.set_ylim([0, 1])
    
    plt.tight_layout()
    
    # Save plot to BytesIO buffer and encode as base64
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plot_base64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close()
    
    # Display plot and video side by side
    if video_filepath and os.path.exists(video_filepath):
        print(f"\n{'='*60}")
        print(f"VIDEO BEING TESTED: {os.path.basename(video_filepath)}")
        print(f"{'='*60}\n")
        
        # Create HTML for side-by-side layout
        html_content = f"""
        <div style="display: flex; flex-wrap: wrap; gap: 20px; align-items: flex-start;">
            <div style="flex: 1; min-width: 500px;">
                <img src="data:image/png;base64,{plot_base64}" style="width: 100%; height: auto;">
            </div>
            <div style="flex: 1; min-width: 500px;">
                <video width="100%" height="auto" controls>
                    <source src="{video_filepath}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
            </div>
        </div>
        """
        display(HTML(html_content))
    else:
        # If no video, just show the plot
        plt.show()
    
    # Print statistics
    print("\n" + "="*60)
    print("DETECTION STATISTICS")
    print("="*60)
    print(f"Total frames analyzed: {len(clip_results)}")
    print(f"Average probability: {sum(probabilities) / len(probabilities):.4f}")
    
    # Count frames above and below threshold
    above_threshold = sum(1 for p in probabilities if p >= threshold)
    below_threshold = len(probabilities) - above_threshold
    
    print(f"\nFrames above threshold ({threshold}): {above_threshold} ({above_threshold/len(probabilities)*100:.1f}%)")
    print(f"Frames below threshold ({threshold}): {below_threshold} ({below_threshold/len(probabilities)*100:.1f}%)")
    
    if final_result:
        verdict = "SYNTHETIC/FAKE" if final_result['probability'] > threshold else "REAL"
        confidence = final_result['probability'] * 100 if final_result['probability'] > threshold else (1 - final_result['probability']) * 100
        
        print("\n" + "*"*60)
        print(f"FINAL VERDICT: {verdict}")
        print(f"CONFIDENCE: {confidence:.2f}%")
        print("*"*60)


print("Visualization function defined successfully!")

## Main Detection Function

Combine all helper functions into a single detection pipeline that analyzes a video and optionally plots the results.


In [ ]:
def detect_synthetic_video(
    video_filepath: Optional[str] = None,
    server_target: str = SERVICE_TARGET,
    save_csv: bool = False,
    output_csv: str = None,
    plot_results: bool = True,
    threshold: float = 0.44,
    presigned_url: Optional[str] = None,
    url_provider: str = "unspecified",
    checksum_type: str = "sha256",
    checksum: Optional[str] = None,
    no_verify_checksum: bool = False,
    expected_size: Optional[int] = None,
    content_type: Optional[str] = None,
    expires_at_unix_ms: Optional[int] = None,
    show_preview: bool = True,
) -> Tuple[List[dict], dict]:
    """Complete pipeline to detect synthetic video content.
    
    Supports both input modes: a local file streamed as bytes, or a pre-signed
    HTTPS URL (S3 ingest) that the NIM downloads itself. Exactly one of
    video_filepath or presigned_url must be supplied.
    
    Args:
        video_filepath: Path to video file to analyze (byte-streaming path)
        server_target: gRPC server address (host:port)
        save_csv: Whether to save detailed CSV results
        output_csv: Path for output CSV file (default: auto-generated)
        plot_results: Whether to plot visualization
        threshold: Threshold for classification (default: 0.44)
        presigned_url: Pre-signed HTTPS URL to the input object (S3 ingest)
        url_provider: 'unspecified' or 's3'
        checksum_type: 'md5', 'sha256', or 'sha512'
        checksum: Expected lowercase hex digest of the object
        no_verify_checksum: Ask the server to skip checksum verification
        expected_size: Expected Content-Length in bytes
        content_type: Expected Content-Type, e.g. 'video/mp4'
        expires_at_unix_ms: Client assertion of URL expiry (epoch ms)
        show_preview: Render an inline preview of the input before inference
    
    Returns:
        Tuple of (clip_results, final_result)
    """
    
    # Label used for headers, CSV names and plot titles in both input modes.
    # Only the redacted URL is printed so signed credentials stay out of output.
    input_label = redact_url(presigned_url) if presigned_url else video_filepath
    
    print("="*60)
    print("SYNTHETIC VIDEO DETECTOR")
    print("="*60)
    print(f"{'URL' if presigned_url else 'Video'}: {input_label}")
    print(f"Server: {server_target}")
    print("="*60)
    print()
    
    # Show what is about to be analyzed (streams from the URL in the browser)
    if show_preview:
        preview_input_video(video_filepath=video_filepath, presigned_url=presigned_url)
        print()
    
    # Create gRPC channel
    with grpc.insecure_channel(server_target) as channel:
        # Create stub
        stub = syntheticvideodetector_pb2_grpc.SyntheticVideoDetectorServiceStub(channel)
        
        # Generate requests (byte streaming or pre-signed URL)
        request_generator = generate_request_for_inference(
            video_filepath=video_filepath,
            presigned_url=presigned_url,
            url_provider=url_provider,
            checksum_type=checksum_type,
            checksum=checksum,
            no_verify_checksum=no_verify_checksum,
            expected_size=expected_size,
            content_type=content_type,
            expires_at_unix_ms=expires_at_unix_ms,
        )
        
        # Call service and process responses
        response_iter = stub.DetectSyntheticVideo(request_generator)
        clip_results, final_result = process_detection_response(response_iter)
    
    print("\n" + "="*60)
    print("DETECTION COMPLETE")
    print("="*60)
    
    # Print results
    if final_result:
        print(f"\nTotal frames processed: {final_result['total_frames']}")
        print(f"Final logit: {final_result['logit']:.6f}")
        print(f"Final probability: {final_result['probability']:.6f}")
        
        verdict = "SYNTHETIC/FAKE" if final_result['probability'] > threshold else "REAL"
        confidence = final_result['probability'] * 100 if final_result['probability'] > threshold else (1 - final_result['probability']) * 100
        
        print(f"\nVERDICT: {verdict} (confidence: {confidence:.2f}%)")
        
        # Save CSV if requested (write index,probability from per-frame results)
        if save_csv and clip_results:
            if output_csv is None:
                # Derive the base name from the URL path when using S3 ingest
                if presigned_url:
                    base = os.path.basename(urlsplit(presigned_url).path) or "presigned_input"
                else:
                    base = os.path.basename(video_filepath)
                video_name = os.path.splitext(base)[0]
                output_csv = f"detection_results_{video_name}.csv"

            import csv
            with open(output_csv, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['index', 'probability'])
                for clip in clip_results:
                    writer.writerow([clip['index'], f"{clip['probability']:.6f}"])
            print(f"\nCSV results saved to: {output_csv}")
    
    # Plot results if requested
    if plot_results and clip_results:
        if presigned_url:
            video_name = os.path.basename(urlsplit(presigned_url).path) or "pre-signed URL"
        else:
            video_name = os.path.basename(video_filepath)
        plot_detection_results(clip_results, final_result, video_filepath=video_filepath, threshold=threshold, video_name=video_name)
    
    return clip_results, final_result


print("Main detection function defined successfully!")

## Example Usage

Now you can use the detection function to analyze videos. Update the video path below to analyze your own videos.


In [ ]:
# Example 1: Analyze a local video file (byte streaming) with visualization
# and the default threshold (0.44, balanced)
VIDEO_PATH = "../../assets/fake_sample_video.mp4"  # Update with your video path

frame_results, final_result = detect_synthetic_video(
    video_filepath=VIDEO_PATH,
    server_target=SERVICE_TARGET,
    plot_results=True,
    threshold=0.44
)

## Example: Pre-signed URL (S3 Ingest)

Instead of uploading the video, you can hand the service a **pre-signed HTTPS URL** and let the NIM fetch the object itself. This keeps large files off the gRPC channel entirely.

A preview of the input is rendered inline as soon as the URL is passed, so you can confirm you are analyzing the right clip before inference starts.

All URL validation (HTTPS, host allowlist, expiry, SSRF, checksum, size, content type) is performed **server-side** by the NIM, so nothing needs to be checked here.

> **Note:** the full signed URL is embedded in the rendered preview. Clear the cell output before sharing or committing this notebook.

In [ ]:
# Example 2: Analyze a video from a pre-signed URL (S3 ingest)
# Paste a pre-signed HTTPS GET URL below. The NIM downloads the object itself,
# so nothing is uploaded from this notebook.
PRESIGNED_URL = "https://<bucket>.s3.<region>.amazonaws.com/<key>.mp4?X-Amz-Algorithm=..."

frame_results, final_result = detect_synthetic_video(
    presigned_url=PRESIGNED_URL,
    server_target=SERVICE_TARGET,
    url_provider="s3",        # enables SigV4 URL-shape checks server-side
    no_verify_checksum=True,  # or pass checksum="<hex>" to have the NIM verify it
    plot_results=True,
    threshold=0.44,
)

## Understanding the Visualization

### 1. Probability Over Time (Left Plot)
- **Blue line**: Per-frame probability score (higher = more likely synthetic)
- **Red dashed line**: Classification threshold (default: 0.44)
- **Green horizontal line**: Final aggregated probability
- **Red shaded area**: Regions where frames are classified as SYNTHETIC (above threshold)
- **Green shaded area**: Regions where frames are classified as REAL (below threshold)

### 2. Video Display (Right Side)
- **Video Player**: The analyzed video is displayed for visual review
- **Playback Controls**: Use the video player to review specific segments
- **Visual Inspection**: Compare the video content with the detection scores to understand which parts may be synthetic

### Key Insights:
- **Consistent high probabilities**: Likely a synthetic video throughout
- **Consistent low probabilities**: Likely a real video throughout
- **Mixed probabilities**: May indicate:
  - Partially synthetic video (some frames edited)
  - Scene changes affecting detection
  - Lower confidence regions requiring manual review

### Threshold Selection:
- **0.44 (default, balanced)**: The balanced V2 operating point, giving an equal trade-off between correctly identifying synthetic and real content.
- **0.26 (conservative)**: The conservative V2 operating point, which more aggressively classifies synthetic videos correctly as synthetic (at the cost of more false positives on real content).
